# 🤿 Sea College Dive Club — Google Colab AI Marketing & Sales Command Center
**Powered by `google.colab.ai` (Zero API Keys Required!) + Live 3-Sheet Google Workspace Sync**

Welcome to your live **Google Colab AI Marketing & Revenue Operations Notebook**, connected directly to your Google Account, `google.colab.ai` Gemini models, and your 3 live Sea College spreadsheets:

1. **[Marketing & Sales Operations Hub](https://docs.google.com/spreadsheets/d/15vrkKvIO9dGbIKOOvHy2MtSf7uKViyvSq8PWb4EINMY/edit)** *(9 Tabs: KPI Scorecard, Accounts, 8-Language URLs, Cutover SOP, Video Calendar, WhatsApp SOP, Competitors, 90-Day Plan & Ready-to-Launch Ads)*
2. **[Website Leads & Offline Conversions CRM](https://docs.google.com/spreadsheets/d/1ItvQGWnrHwnSbyGDoGNtV2CkVwNJXIyU_aaIFxIOGbc/edit)** *(Live Website Leads, Diver Check-Ins & `Google_Ads_OCT_Export`)*
3. **[Google Ads Live Sync Hub (`365-126-5545`)](https://docs.google.com/spreadsheets/d/15IkDV14kxGAbr8DSGOB6auNe4paTNOa5UXmw9d5kVtk/edit)** *(Live Hourly Campaigns, Keywords, Search Terms & Ads)*
4. **[Master Digital Marketing & 90-Day Activation Playbook (Google Doc)](https://docs.google.com/document/d/1V_mQi3HSHKoONtyNTVrU7sSYPu89oqz-0ZcrwWVMLOc/edit)**

### How to Use (Takes 30 Seconds)
* Click **`Runtime -> Run all`** in the top menu (or click the **▶ Play** button on Steps 1 → 8 in order).
* In **Step 2**, click **Allow** when Google Colab asks to connect to your Google Drive / Google Sheets.
* **Zero API keys or billing setup needed:** Step 1 uses `from google.colab import ai` directly from your signed-in Colab account!

In [ ]:
# @title 1. Connect to `google.colab.ai` & Auto-Select Best Available Gemini Model { display-mode: "form" }
# @markdown Lists all models unlocked on your Colab account (`ai.list_models()`) and selects your strongest **Gemini Pro** model automatically (or **Gemini Flash** on free tier).
preferred_tier = "auto_best_pro" # @param ["auto_best_pro", "auto_fast_flash"]

import datetime
import json
import textwrap
from IPython.display import Markdown, display

try:
    from google.colab import ai
    AVAILABLE_MODELS = list(ai.list_models())
    print(f"✅ Connected to google.colab.ai! Found {len(AVAILABLE_MODELS)} models on your account:")
    for m in AVAILABLE_MODELS:
        print(f"   • {m}")
except Exception as e:
    AVAILABLE_MODELS = []
    raise RuntimeError(
        "Run this notebook inside https://colab.research.google.com/ so `google.colab.ai` is active! Original error: " + str(e)
    )

def pick_best_model(models, mode="auto_best_pro"):
    if not models:
        return None
    lower_map = [(m, m.lower()) for m in models]
    if mode == "auto_best_pro":
        # Prefer newest Gemini Pro model if available on the user's Google AI / Colab plan
        pro_candidates = [m for m, lw in lower_map if "gemini" in lw and "pro" in lw]
        if pro_candidates:
            return sorted(pro_candidates)[-1]
    flash_candidates = [m for m, lw in lower_map if "gemini" in lw and "flash" in lw and "lite" not in lw]
    if flash_candidates:
        return sorted(flash_candidates)[-1]
    gemini_any = [m for m, lw in lower_map if "gemini" in lw]
    return sorted(gemini_any)[-1] if gemini_any else models[0]

SELECTED_MODEL = pick_best_model(AVAILABLE_MODELS, preferred_tier)
print(f"\n🚀 Active Marketing Model Selected: {SELECTED_MODEL}")

def ask_gemini_colab(prompt: str, model_name: str = None, stream: bool = True, render_markdown: bool = True) -> str:
    """Calls google.colab.ai with streaming and automatic default-model fallback."""
    chosen = model_name or SELECTED_MODEL
    chunks = []
    try:
        if stream:
            resp_stream = ai.generate_text(prompt, model_name=chosen, stream=True) if chosen else ai.generate_text(prompt, stream=True)
            for chunk in resp_stream:
                chunks.append(str(chunk))
                if not render_markdown:
                    print(chunk, end="", flush=True)
            full_text = "".join(chunks)
        else:
            full_text = ai.generate_text(prompt, model_name=chosen) if chosen else ai.generate_text(prompt)
    except Exception:
        # Clean fallback to default model if a tier-restricted model name was selected
        full_text = ai.generate_text(prompt)

    if render_markdown:
        display(Markdown(full_text))
    return full_text

In [ ]:
# @title 2. Authenticate Google Account & Sync Live Data Across Your 3 Google Sheets { display-mode: "form" }
# @markdown Connects Colab directly to your **Marketing & Sales Hub**, **Website Leads CRM**, and **Google Ads Sync Hub**, and updates `01_Executive_KPI_Hub` (`Month 1 Actual`) with live numbers!
update_kpi_dashboard_now = True # @param {type:"boolean"}

from google.colab import auth
import google.auth
import gspread
import pandas as pd

auth.authenticate_user()
creds, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
])
gc = gspread.authorize(creds)

SHEET_IDS = {
    "MKT_HUB": "15vrkKvIO9dGbIKOOvHy2MtSf7uKViyvSq8PWb4EINMY",
    "LEADS_CRM": "1ItvQGWnrHwnSbyGDoGNtV2CkVwNJXIyU_aaIFxIOGbc",
    "ADS_SYNC": "15IkDV14kxGAbr8DSGOB6auNe4paTNOa5UXmw9d5kVtk",
}
EUR_TO_EGP = 52.0

mkt_wb = gc.open_by_key(SHEET_IDS["MKT_HUB"])
crm_wb = gc.open_by_key(SHEET_IDS["LEADS_CRM"])
ads_wb = gc.open_by_key(SHEET_IDS["ADS_SYNC"])

print("✅ Connected to all 3 Live Sea College Spreadsheets:")
print(f"   1. {mkt_wb.title} -> {[ws.title for ws in mkt_wb.worksheets()]}")
print(f"   2. {crm_wb.title} -> {[ws.title for ws in crm_wb.worksheets()]}")
print(f"   3. {ads_wb.title} -> {[ws.title for ws in ads_wb.worksheets()]}")

# Pull live CRM Leads & Paid Revenue
leads_rows = crm_wb.worksheet("Leads").get_all_values()
leads_df = pd.DataFrame(leads_rows[1:], columns=leads_rows[0]) if len(leads_rows) > 1 else pd.DataFrame(columns=leads_rows[0] if leads_rows else [])

total_leads = len(leads_df)
paid_bookings = 0
paid_revenue_eur = 0.0
if not leads_df.empty and "Status" in leads_df.columns:
    paid_mask = leads_df["Status"].astype(str).str.strip().str.lower() == "paid"
    paid_bookings = int(paid_mask.sum())
    if "Final_Paid_EUR" in leads_df.columns:
        paid_revenue_eur = pd.to_numeric(leads_df.loc[paid_mask, "Final_Paid_EUR"], errors="coerce").fillna(0).sum()

# Pull live 30-day Google Ads Performance
camp_rows = ads_wb.worksheet("Campaigns").get_all_values()
camp_df = pd.DataFrame(camp_rows[1:], columns=camp_rows[0]) if len(camp_rows) > 1 else pd.DataFrame()

spend_30d_egp = pd.to_numeric(camp_df.get("Cost_30d_EGP", pd.Series(dtype=float)), errors="coerce").fillna(0).sum() if not camp_df.empty else 0.0
clicks_30d = int(pd.to_numeric(camp_df.get("Clicks_30d", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not camp_df.empty else 0
impr_30d = int(pd.to_numeric(camp_df.get("Impr_30d", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not camp_df.empty else 0
conv_30d = float(pd.to_numeric(camp_df.get("Conv_30d", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if not camp_df.empty else 0.0
spend_30d_eur = round(spend_30d_egp / EUR_TO_EGP, 2)

LIVE_SNAPSHOT = {
    "synced_at_utc": datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
    "crm_total_leads": total_leads,
    "crm_paid_bookings": paid_bookings,
    "crm_paid_revenue_eur": round(float(paid_revenue_eur), 2),
    "google_ads_30d_impressions": impr_30d,
    "google_ads_30d_clicks": clicks_30d,
    "google_ads_30d_spend_egp": round(float(spend_30d_egp), 2),
    "google_ads_30d_spend_eur": spend_30d_eur,
    "google_ads_30d_recorded_conversions": conv_30d,
}

print("\n📊 LIVE PERFORMANCE SNAPSHOT:")
for k, v in LIVE_SNAPSHOT.items():
    print(f"   • {k}: {v}")

if update_kpi_dashboard_now:
    kpi_ws = mkt_wb.worksheet("01_Executive_KPI_Hub")
    aov_actual = round(paid_revenue_eur / paid_bookings, 2) if paid_bookings > 0 else 340.0
    close_rate = round(paid_bookings / total_leads, 4) if total_leads > 0 else 0.22
    kpi_ws.update(
        range_name="F10:F17",
        values=[
            [spend_30d_eur],
            [0],
            [total_leads],
            ["=IFERROR(F10/F12,0)"],
            [5],
            [close_rate],
            [paid_bookings],
            [aov_actual],
        ],
        value_input_option="USER_ENTERED"
    )
    print("\n✅ Updated `01_Executive_KPI_Hub` (Month 1 Actuals) inside your Marketing & Sales Google Sheet!")

In [ ]:
# @title 3. Load Complete Sea College Website, 8-Language URLs, EUR Catalog & Competitor Brain { display-mode: "form" }
# @markdown Loads 100% of verified Sea College Dive Club intelligence (80 item pages, EUR prices, GBP `4.9★ · 3,082` reviews, Tripadvisor `#26104236`, tracking IDs, and competitor benchmarks) into memory for all AI generators below.

SEA_COLLEGE_BRAIN = json.loads("{\n  \"prompt_metadata\": {\n    \"schema_version\": \"2026.11-final-live\",\n    \"project\": \"Sea College Dive Club \u2014 Master Digital Marketing, Sales & RevOps Execution Plan\",\n    \"recommended_gemini_models\": {\n      \"primary_strategic_and_financial_planner\": \"Gemini 4 Carbon (Google AI Ultra / AI Studio) OR Gemini 4 Argon / Gemini 3.1 Pro (Google AI Pro / Workspace) with Canvas ON\",\n      \"live_market_validation_mode\": \"Gemini Deep Research (to cross-check live seasonal flight & search trends into Sharm El Sheikh SSH across UK, DE, IT, PL, RU, FR, GCC, CN)\",\n      \"daily_execution_and_multilingual_copy\": \"Custom Gemini Gem ('Sea College Growth Director') powered by Gemini 3.8 Flash or Gemini 4 Argon with this JSON attached in Knowledge + @Google Workspace enabled\"\n    },\n    \"system_role_instruction\": \"Act as an elite Chief Marketing Officer (CMO), Red Sea Dive Tourism Performance Strategist, Multilingual Google/Meta Ads Architect, SEO/AEO Director, and WhatsApp/CRM Revenue Operations Lead. Using ONLY the verified business, website, pricing, tracking, CRM, competitor, and historical Google Ads data in this JSON, produce a 100% applicable, zero-fluff, week-by-week Digital Marketing & Sales Plan + 90-Day Action Plan engineered to maximize direct 0%-commission bookings from our new 8-language website and social platforms, tightly integrated with our 3 live Google Sheets.\",\n    \"strict_grounding_rules\": [\n      \"Never invent fake prices, fake URLs, or unverified claims. Use the exact EUR prices, clean slugs, and tracking IDs below.\",\n      \"Every paid or organic campaign MUST route users to the matching language subdirectory (/ru/..., /it/..., /pl/..., /de/..., /fr/..., /ar/..., /zh/..., or default English /...) and append clean UTM parameters.\",\n      \"Respect the transparent Red Sea pricing model: state base package EUR prices clearly and disclose mandatory daily add-ons (\u20ac10/day boat supplement, \u20ac5/day Ras Mohammed/Tiran marine park fee, \u20ac25/day optional full equipment rental, \u20ac8/day 12L Nitrox) using our Interactive Dive Budget Calculator as a trust weapon against bait-and-switch street/OTA brokers.\",\n      \"Tie every marketing channel to our closed-loop CRM workflow: Website Form / Calculator / Quiz / WhatsApp -> Website Leads & OCT CRM Sheet -> <5 min WhatsApp Sales SOP -> Status='Paid' + Final_Paid_EUR -> Google_Ads_OCT_Export -> Google Ads Smart Bidding.\",\n      \"Make every recommendation realistic for a lean, high-output dive center team starting at ~$2,000/month (\u20ac1,850/mo) paid media budget and scaling aggressively as verified ROAS exceeds 5.0x.\"\n    ]\n  },\n  \"business_profile\": {\n    \"brand_name\": \"Sea College Dive Club\",\n    \"accreditations\": [\n      \"Official PADI 5 Star Dive Resort\",\n      \"PADI 'Adopt the Blue' Marine Conservation Partner\",\n      \"CDWS (Chamber of Diving and Water Sports \u2014 Egyptian Ministry of Tourism) Accredited Center\"\n    ],\n    \"founded\": 2019,\n    \"location\": {\n      \"area\": \"Al Montazah Beachfront (facing Tiran Island / Straits of Tiran)\",\n      \"host_resorts\": \"On-site dive center inside Cyrene Sharm Hotel / Ivy Cyrene Island Aqua Park Resort\",\n      \"city\": \"Sharm El Sheikh, South Sinai, Egypt\",\n      \"coordinates\": {\n        \"lat\": 27.9805263,\n        \"lng\": 34.4239091\n      },\n      \"operating_hours\": \"Mon-Sun 08:00-18:00 (WhatsApp reservations active 08:00-22:00 Cairo time)\",\n      \"logistics_advantage\": \"Direct beachfront House Reef + private on-site training pool + 5-10 min transfer to Sharks Bay jetty for Tiran boats + free AC hotel pickup/drop-off across Sharm El Sheikh.\"\n    },\n    \"reputation_and_social_proof\": {\n      \"google_business_profile\": {\n        \"rating\": 4.9,\n        \"verified_reviews_count\": 3082,\n        \"place_id\": \"ChIJ75cBQp5LUxQRMjrNYQx93g0\",\n        \"maps_url\": \"https://maps.app.goo.gl/JweBzniP2hS1VpgK7\",\n        \"direct_write_review_url\": \"https://search.google.com/local/writereview?placeid=ChIJ75cBQp5LUxQRMjrNYQx93g0\"\n      },\n      \"tripadvisor_official\": {\n        \"rating\": 4.9,\n        \"verified_reviews_count\": 129,\n        \"location_id\": \"26104236\",\n        \"widget_ids\": {\n          \"en_US\": \"409\",\n          \"ar_EG\": \"969\"\n        },\n        \"listing_url\": \"https://www.tripadvisor.com/Attraction_Review-g297555-d26104236-Reviews-Sea_college_dive_club-Sharm_El_Sheikh_South_Sinai_Red_Sea_and_Sinai.html\",\n        \"direct_write_review_url\": \"https://www.tripadvisor.com/UserReview-g297555-d26104236-Sea_college_dive_club-Sharm_El_Sheikh_South_Sinai_Red_Sea_and_Sinai.html\",\n        \"owner_account_note\": \"Managed under Seacollegediveclub2019@gmail.com; live review cards & regional domain switching integrated across all 8 website languages.\"\n      },\n      \"named_instructors_praised_in_reviews\": [\n        \"Mo (patient beginner DSD & Open Water instructor)\",\n        \"Sam (PADI courses & calm confidence building)\",\n        \"Jimmy / \u0414\u0436\u0438\u043c\u0438 (top-praised Russian & English instructor)\",\n        \"Moussab (guided boat & wreck dives)\",\n        \"Alex (European & advanced divers)\",\n        \"Wesam / \u0648\u0633\u0627\u0645 & Saleh / \u0635\u0627\u0644\u062d (top-praised Arabic & beginner instructors)\",\n        \"Mazen, Emad & Mustafa (safety, boat crew & underwater photography)\"\n      ],\n      \"core_unique_selling_propositions\": [\n        \"#1 verified social proof combination in Al Montazah/Sharks Bay: 4.9\u2605 across 3,082+ Google reviews + 4.9\u25cf Tripadvisor.\",\n        \"\u20ac0 upfront credit card fee / zero-risk online reservation (reserve spots now, pay at the dive center in EUR, USD, GBP, EGP or card).\",\n        \"On-site private training pool + calm sandy-entry House Reef (eliminates scary deep-water first breaths for beginners and kids aged 8+).\",\n        \"100% transparent EUR pricing with an interactive Dive Budget Calculator (contrasts sharply with Sharm street/OTA bait pricing that hides boat, park, and gear surcharges).\",\n        \"Multilingual PADI 5\u2605 team + 8-language native website (EN, RU, IT, PL, DE, FR, AR, ZH).\"\n      ]\n    },\n    \"contacts_and_official_channels\": {\n      \"primary_whatsapp_and_phone\": \"+201000344544\",\n      \"whatsapp_deep_link\": \"https://wa.me/201000344544\",\n      \"backup_phones\": [\n        \"+201114567886\",\n        \"+201006145500\"\n      ],\n      \"reservations_email\": \"info@seacollegediveclub.com\",\n      \"youtube\": \"https://www.youtube.com/@seacollege.diveclub\",\n      \"instagram\": \"https://www.instagram.com/seacollege.diveclub/\",\n      \"facebook\": \"https://www.facebook.com/seacollegediveclub\",\n      \"tiktok\": \"https://www.tiktok.com/@seacollegediveclub\",\n      \"padi_locator\": \"https://www.padi.com/dive-center/egypt/sea-college-dive-club/\",\n      \"cdws_listing\": \"https://www.diveoneclub.com/Listing/sea-college-dive-club/\",\n      \"existing_ota_listings_to_outperform_directly\": [\n        \"Peek.com\",\n        \"Trip.com (20%-25% commission cost)\"\n      ]\n    }\n  },\n  \"final_live_website_capabilities\": {\n    \"production_domain\": \"https://seacollegediveclub.com\",\n    \"staging_url_verified\": \"https://papayawhip-pheasant-312330.hostingersite.com\",\n    \"tech_stack\": \"Fast, zero-bloat custom PHP 8 + Vanilla JS + Apache/LiteSpeed .htaccess rewrite engine (<1s load, 0 WooCommerce/WordPress plugin bloat, mobile-first Android/iOS/Desktop UI)\",\n    \"supported_languages_in_header_order\": [\n      {\n        \"code\": \"en\",\n        \"name\": \"English\",\n        \"url_prefix\": \"https://seacollegediveclub.com/\"\n      },\n      {\n        \"code\": \"ru\",\n        \"name\": \"\u0420\u0443\u0441\u0441\u043a\u0438\u0439\",\n        \"url_prefix\": \"https://seacollegediveclub.com/ru/\"\n      },\n      {\n        \"code\": \"it\",\n        \"name\": \"Italiano\",\n        \"url_prefix\": \"https://seacollegediveclub.com/it/\"\n      },\n      {\n        \"code\": \"pl\",\n        \"name\": \"Polski\",\n        \"url_prefix\": \"https://seacollegediveclub.com/pl/\"\n      },\n      {\n        \"code\": \"de\",\n        \"name\": \"Deutsch\",\n        \"url_prefix\": \"https://seacollegediveclub.com/de/\"\n      },\n      {\n        \"code\": \"fr\",\n        \"name\": \"Fran\u00e7ais\",\n        \"url_prefix\": \"https://seacollegediveclub.com/fr/\"\n      },\n      {\n        \"code\": \"ar\",\n        \"name\": \"\u0627\u0644\u0639\u0631\u0628\u064a\u0629 (RTL)\",\n        \"url_prefix\": \"https://seacollegediveclub.com/ar/\"\n      },\n      {\n        \"code\": \"zh\",\n        \"name\": \"\u4e2d\u6587\",\n        \"url_prefix\": \"https://seacollegediveclub.com/zh/\"\n      }\n    ],\n    \"seo_and_aeo_architecture\": {\n      \"xml_sitemap_url\": \"https://seacollegediveclub.com/sitemap.xml\",\n      \"total_indexed_urls\": 744,\n      \"routes_per_language\": 93,\n      \"dedicated_item_pages\": 80,\n      \"category_hub_urls\": [\n        \"/diving-trips-sharm-el-sheikh\",\n        \"/diving-courses-sharm-el-sheikh\",\n        \"/padi-course-bundles-sharm-el-sheikh\",\n        \"/snorkeling-and-boats-sharm-el-sheikh\"\n      ],\n      \"authority_content_hubs\": [\n        \"/dive-sites-sharm-el-sheikh (Interactive Red Sea Map + Depth/Certification Cards + 12-Month Water Temp & Visibility Chart)\",\n        \"/gallery (Authentic Underwater Photo/Video Lightbox + Named PADI Instructor Profiles)\",\n        \"/prices.php (Full EUR Price List + Interactive Dive Budget Calculator + Online Diver Check-In)\",\n        \"/about.php, /services.php, /contact.php\"\n      ],\n      \"structured_data_json_ld\": [\n        \"SportsActivityLocation + LocalBusiness + DiveCenter (NAP, GeoCoordinates, 4.9\u2605 AggregateRating 3,082 reviews, sameAs social/PADI/Tripadvisor)\",\n        \"Product + Offer (EUR price, InStock, priceValidUntil) on all 80 item pages\",\n        \"Course (PADI courses) & TouristTrip (dive/snorkeling/boat tours)\",\n        \"VideoObject (embedded HD underwater video galleries)\",\n        \"FAQPage (AEO direct-answer Q&As for ChatGPT, Gemini AI Overviews & Perplexity)\",\n        \"BreadcrumbList & bidirectional 8-language + x-default hreflang clusters\"\n      ],\n      \"legacy_url_protection\": \"All old WordPress /product/<slug>/ URLs 301-redirect cleanly to the new /tours/<slug> or /courses/<slug> while preserving ?gclid, ?fbclid, and ?utm_* via [L,QSA].\"\n    },\n    \"interactive_conversion_tools_on_site\": [\n      {\n        \"tool\": \"Interactive Dive Budget & Fee Calculator\",\n        \"url\": \"https://seacollegediveclub.com/prices.php#calculator\",\n        \"mechanism\": \"Divers select number of dives (1-10+) + boat days (\u20ac10/day) + marine park days (\u20ac5/day) + full equipment rental (\u20ac25/day) + Nitrox (\u20ac8/day) -> pushes exact itemized bundle into Cart or pre-filled WhatsApp.\"\n      },\n      {\n        \"tool\": \"30-Second 'Find My Dive / Course' Recommender Quiz\",\n        \"url\": \"https://seacollegediveclub.com/#quiz\",\n        \"mechanism\": \"Matches undecided tourists (certification status, water comfort, days in Sharm, group type) to the ideal Intro Dive, PADI Open Water, Mega Bundle, or Private Boat.\"\n      },\n      {\n        \"tool\": \"Interactive Sharm Dive Sites & Monthly Conditions Guide\",\n        \"url\": \"https://seacollegediveclub.com/dive-sites-sharm-el-sheikh\",\n        \"mechanism\": \"Covers Ras Mohammed (Shark & Yolanda Reef), Straits of Tiran (Jackson, Woodhouse, Thomas, Gordon), SS Thistlegorm, SS Dunraven, Million Hope & Al Montazah House Reef + monthly temp (22\u00b0C-29\u00b0C) & visibility (25m-35m).\"\n      },\n      {\n        \"tool\": \"Multi-Item Booking Cart + Dual Checkout + Pre-Arrival Diver Check-In\",\n        \"url\": \"https://seacollegediveclub.com/prices.php#checkin\",\n        \"mechanism\": \"Visitors add courses/dives/supplements to persistent cart -> submit instant Reservation Form (writes to CRM + fires Enhanced Conversions) OR send itemized cart to WhatsApp (+20 100 034 4544). Confirmed guests complete Online PADI Check-In prior to arrival.\"\n      }\n    ],\n    \"analytics_and_ad_tracking\": {\n      \"gtm_container_id\": \"GTM-PBVQ56XC\",\n      \"ga4_measurement_id\": \"G-99528RZN9Z\",\n      \"google_ads_tag_id\": \"AW-16728866067\",\n      \"google_ads_conversion_labels\": {\n        \"reservation_form_lead\": \"LCPWCKD0vOIcEJP6-Kg-\",\n        \"whatsapp_lead_click\": \"rh4WCOXW4d8cEJP6-Kg-\",\n        \"phone_call_click\": \"8hA2CLrgqd4cEJP6-Kg-\"\n      },\n      \"meta_pixel_id\": \"2980406819004765 (Browser Pixel + Server-Side Meta Conversions API with shared Event_ID deduplication)\",\n      \"attribution_cookie_persistence\": \"90-day first/last-touch cookie storing utm_source, utm_medium, utm_campaign, utm_term, utm_content, gclid, gbraid, wbraid, fbclid, ttclid\"\n    }\n  },\n  \"official_eur_pricing_catalog\": {\n    \"currency_note\": \"Primary currency is EUR (display switcher supports EUR, USD, GBP, EGP). \u20ac0 upfront online booking fee.\",\n    \"beginner_intro_dives\": [\n      {\n        \"name\": \"Quick Discover Scuba Diving \u2014 House Reef (1 shore dive)\",\n        \"price_eur\": 30,\n        \"slug\": \"/tours/quick-discover-scuba-diving-house-reef\"\n      },\n      {\n        \"name\": \"Discover Scuba Diving \u2014 House Reef (2 shore dives)\",\n        \"price_eur\": 75,\n        \"slug\": \"/tours/discover-scuba-diving-2-dives-house-reef\"\n      },\n      {\n        \"name\": \"Quick Discover Scuba Diving \u2014 By Boat (1 dive, all-inclusive)\",\n        \"price_eur\": 70,\n        \"slug\": \"/tours/quick-discover-scuba-diving-boat-sharm\"\n      },\n      {\n        \"name\": \"Discover Scuba Diving \u2014 By Boat (2 dives, lunch, park & gear included)\",\n        \"price_eur\": 100,\n        \"slug\": \"/tours/discover-scuba-diving-2-dives-boat-sharm\"\n      }\n    ],\n    \"refresher_and_certified_dive_packages\": [\n      {\n        \"name\": \"Check Dive\",\n        \"price_eur\": 25,\n        \"slug\": \"/tours/check-dive-refresher-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Scuba Review\",\n        \"price_eur\": 50,\n        \"slug\": \"/tours/scuba-review-refresher-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Single Shore Dive\",\n        \"price_eur\": 45,\n        \"slug\": \"/tours/single-shore-dive-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"2 Dives Package (Boat or Shore)\",\n        \"price_eur\": 85,\n        \"slug\": \"/tours/2-dives-boat-or-shore-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"4 Dives Package\",\n        \"price_eur\": 140,\n        \"slug\": \"/tours/4-dives-package-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"6 Dives Package (3 days)\",\n        \"price_eur\": 210,\n        \"slug\": \"/tours/6-dives-package-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"8 Dives Package (4 days)\",\n        \"price_eur\": 280,\n        \"slug\": \"/tours/8-dives-package-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"10 Dives Package (5 days \u2014 \u20ac35/dive)\",\n        \"price_eur\": 350,\n        \"slug\": \"/tours/10-dives-package-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Extra Dive after 10th\",\n        \"price_eur\": 30,\n        \"slug\": \"/tours/extra-dive-after-10-dives-sharm\"\n      }\n    ],\n    \"wrecks_zodiac_and_mandatory_daily_supplements\": [\n      {\n        \"name\": \"Boat Trip Daily Supplement\",\n        \"price_eur\": 10,\n        \"unit\": \"per day\",\n        \"slug\": \"/tours/boat-trip-daily-supplement-sharm\"\n      },\n      {\n        \"name\": \"Ras Mohammed / Tiran Marine Park Fee\",\n        \"price_eur\": 5,\n        \"unit\": \"per day\",\n        \"slug\": \"/tours/ras-mohammed-tiran-marine-park-fee\"\n      },\n      {\n        \"name\": \"Full Scuba Equipment Rental Set\",\n        \"price_eur\": 25,\n        \"unit\": \"per day\",\n        \"slug\": \"/tours/full-scuba-equipment-rental-sharm\"\n      },\n      {\n        \"name\": \"12L Enriched Air Nitrox Fill\",\n        \"price_eur\": 8,\n        \"unit\": \"per tank/day\",\n        \"slug\": \"/tours/12l-nitrox-tank-fill-sharm\"\n      },\n      {\n        \"name\": \"Private Dive Guide\",\n        \"price_eur\": 75,\n        \"unit\": \"per day\",\n        \"slug\": \"/tours/private-dive-guide-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"SS Thistlegorm WWII Wreck Supplement\",\n        \"price_eur\": 150,\n        \"slug\": \"/tours/ss-thistlegorm-wreck-dive-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"SS Dunraven or Million Hope Wreck Supplement\",\n        \"price_eur\": 30,\n        \"slug\": \"/tours/ss-dunraven-wreck-dive-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Zodiac RIB Speedboat Dive\",\n        \"price_eur\": 50,\n        \"slug\": \"/tours/zodiac-speedboat-dive-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Early Morning Zodiac RIB Dive (Hammerheads)\",\n        \"price_eur\": 70,\n        \"slug\": \"/tours/early-morning-zodiac-dive-hammerheads-sharm\"\n      },\n      {\n        \"name\": \"Dahab Dive Trip from Sharm (Blue Hole & Canyon)\",\n        \"price_eur\": 115,\n        \"slug\": \"/tours/dahab-dive-trip-from-sharm-el-sheikh\"\n      }\n    ],\n    \"padi_courses_and_value_bundles\": [\n      {\n        \"name\": \"PADI Bubblemaker (Kids 8+)\",\n        \"price_eur\": 70,\n        \"slug\": \"/courses/padi-bubblemaker-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Scuba Diver (2 days)\",\n        \"price_eur\": 250,\n        \"slug\": \"/courses/padi-scuba-diver-course-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Open Water Diver (3-4 days)\",\n        \"price_eur\": 380,\n        \"slug\": \"/courses/padi-open-water-diver-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Advanced Open Water Diver (2 days, 5 dives)\",\n        \"price_eur\": 280,\n        \"slug\": \"/courses/padi-advanced-open-water-diver-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Enriched Air Nitrox Course\",\n        \"price_eur\": 145,\n        \"slug\": \"/courses/padi-enriched-air-nitrox-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Rescue Diver (\u20ac394) + EFR (\u20ac143)\",\n        \"price_eur\": 537,\n        \"slug\": \"/courses/padi-rescue-diver-course-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Deep Diver or Wreck Diver Specialty\",\n        \"price_eur\": 325,\n        \"slug\": \"/courses/padi-deep-diver-course-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"PADI Divemaster Internship\",\n        \"price_eur\": 850,\n        \"slug\": \"/courses/padi-divemaster-internship-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"STARS Bundle (Open Water + Nitrox)\",\n        \"price_eur\": 500,\n        \"slug\": \"/courses/stars-bundle-open-water-nitrox-sharm\"\n      },\n      {\n        \"name\": \"SUPER Bundle (Advanced Open Water + Nitrox)\",\n        \"price_eur\": 400,\n        \"slug\": \"/courses/super-bundle-advanced-open-water-nitrox\"\n      },\n      {\n        \"name\": \"MEGA Bundle (Open Water + Advanced Open Water)\",\n        \"price_eur\": 600,\n        \"slug\": \"/courses/mega-bundle-open-water-advanced-sharm\"\n      },\n      {\n        \"name\": \"HEROS Bundle (Open Water + Advanced + Nitrox)\",\n        \"price_eur\": 750,\n        \"slug\": \"/courses/heros-bundle-open-water-advanced-nitrox\"\n      }\n    ],\n    \"snorkeling_speedboats_and_private_yachts\": [\n      {\n        \"name\": \"White Island & Ras Mohammed Boat Snorkeling (Adult / Child)\",\n        \"price_eur\": \"55 / 30\",\n        \"slug\": \"/tours/white-island-ras-mohammed-boat-trip\"\n      },\n      {\n        \"name\": \"House Reef Beach Snorkeling (Day / Night)\",\n        \"price_eur\": \"35 / 40\",\n        \"slug\": \"/tours/house-reef-beach-snorkeling-sharm-el-sheikh\"\n      },\n      {\n        \"name\": \"Private Speedboat Snorkeling (1h Local \u20ac75 / 2h Tiran \u20ac150 / 3h Tiran \u20ac225 per boat)\",\n        \"price_eur\": \"75 - 225\",\n        \"slug\": \"/tours/tiran-island-private-speedboat-2-hours\"\n      },\n      {\n        \"name\": \"Full-Day Private Yacht Charter (1-5 pax \u20ac550 | 6-10 pax \u20ac650 | 11-15 pax \u20ac900 | 16-20 pax \u20ac1,100)\",\n        \"price_eur\": \"550 - 1100\",\n        \"slug\": \"/tours/private-boat-charter-6-10-guests-sharm\"\n      }\n    ]\n  },\n  \"connected_crm_and_operations_spreadsheets\": {\n    \"sheet_1_website_leads_and_oct_crm\": {\n      \"spreadsheet_id\": \"1ItvQGWnrHwnSbyGDoGNtV2CkVwNJXIyU_aaIFxIOGbc\",\n      \"url\": \"https://docs.google.com/spreadsheets/d/1ItvQGWnrHwnSbyGDoGNtV2CkVwNJXIyU_aaIFxIOGbc/edit\",\n      \"tabs\": [\n        \"Leads\",\n        \"Checkins\",\n        \"Newsletter\",\n        \"Google_Ads_OCT_Export\"\n      ],\n      \"automation_flow\": \"Website reservation/cart submissions POST via Apps Script webhook with Event_ID, Cart_JSON, Estimated_EUR, Lang, gclid, fbclid & UTMs. When Reservation Desk updates Status='Paid' and enters Final_Paid_EUR, Google_Ads_OCT_Export automatically formats the GCLID + EUR conversion value for daily Google Ads Offline Conversion import.\"\n    },\n    \"sheet_2_google_ads_live_sync_hub\": {\n      \"spreadsheet_id\": \"15IkDV14kxGAbr8DSGOB6auNe4paTNOa5UXmw9d5kVtk\",\n      \"url\": \"https://docs.google.com/spreadsheets/d/15IkDV14kxGAbr8DSGOB6auNe4paTNOa5UXmw9d5kVtk/edit\",\n      \"tabs\": [\n        \"Account_Overview\",\n        \"Campaigns\",\n        \"Ad_Groups\",\n        \"Keywords\",\n        \"Search_Terms\",\n        \"Ads\",\n        \"Conversion_Actions\",\n        \"Action_Queue\"\n      ],\n      \"automation_flow\": \"Hourly Google Ads Script exports live campaign, keyword, search term, and ad performance from Account 365-126-5545.\"\n    },\n    \"sheet_3_master_marketing_and_sales_operations_hub\": {\n      \"spreadsheet_id\": \"15vrkKvIO9dGbIKOOvHy2MtSf7uKViyvSq8PWb4EINMY\",\n      \"url\": \"https://docs.google.com/spreadsheets/d/15vrkKvIO9dGbIKOOvHy2MtSf7uKViyvSq8PWb4EINMY/edit\",\n      \"tabs\": [\n        \"01_Executive_KPI_Hub (IMPORTRANGE live formulas from Sheets 1 & 2 + Monthly Funnel Targets)\",\n        \"02_Accounts_Tech_Stack\",\n        \"03_8Lang_Ads_URL_Matrix\",\n        \"04_Google_Ads_Cutover_SOP\",\n        \"05_Social_Video_Calendar\",\n        \"06_Sales_CRM_WhatsApp_SOP\",\n        \"07_Competitor_Price_Benchmark\",\n        \"08_90Day_Action_Plan\"\n      ]\n    }\n  },\n  \"live_google_ads_account_baseline_and_gaps\": {\n    \"customer_id\": \"365-126-5545\",\n    \"currency\": \"EGP (approx 52 EGP = 1 EUR)\",\n    \"active_campaign\": \"sea college start (ID: 24058786556, SEARCH, MAXIMIZE_CONVERSIONS, Daily Budget: 2,000 EGP)\",\n    \"all_time_historical\": {\n      \"impressions\": 78307,\n      \"clicks\": 5030,\n      \"ctr_pct\": 6.42,\n      \"spend_egp\": 134423,\n      \"recorded_conversions\": 1755\n    },\n    \"last_30_days_snapshot\": {\n      \"impressions\": 11958,\n      \"clicks\": 779,\n      \"ctr_pct\": 6.51,\n      \"avg_cpc_egp\": 47.1,\n      \"spend_egp\": 36691.73,\n      \"recorded_conversions\": 2,\n      \"broken_cpa_egp\": 18345.87\n    },\n    \"diagnosed_leaks_to_fix_in_plan\": [\n      \"LEAK 1 (Broken Old-Site Tracking): Old WordPress conversion tracking broke recently (only 2 conversions logged across 779 clicks!), starving Smart Bidding of real conversion signals.\",\n      \"LEAK 2 (Single Homepage Dump): Active Ad 817963783396 sends 100% of clicks to root https://seacollegediveclub.com/ instead of dedicated course, dive-package, wreck, or private-boat pages.\",\n      \"LEAK 3 (Multilingual Search Leak): Historical search terms included 8,014 EN, 1,087 AR, 522 RU, and 743 IT/DE/PL queries all landing on English!\",\n      \"LEAK 4 (Ad Copy Typos): Active RSA headlines include 'VIP Private Boot Ras Mohamed' and 'PADI 5 Sare Certified' \u2014 fix in parallel new RSAs without resetting Ad 817963783396 on Day 0.\"\n    ]\n  },\n  \"market_and_competitor_intelligence\": {\n    \"seasonality_sharm_el_sheikh\": {\n      \"peak_seasons\": \"March-May (Spring European Reef & Wreck peak) & September-November (Autumn warmest sea + peak certified divers) + Late Dec/New Year & Easter charters\",\n      \"summer_opportunity_jun_aug\": \"28-29\u00b0C water, calm seas \u2014 peak GCC/Saudi/Egyptian families (Private Yachts, Speedboats, Intro DSDs) + Italian/Polish/UK school-holiday beginners\",\n      \"winter_opportunity_dec_feb\": \"22-23\u00b0C water, sunny air \u2014 peak escaping-winter Europeans (UK, DE, PL, IT) & Russian/CIS resort guests\"\n    },\n    \"competitor_segments\": {\n      \"legacy_premium_dive_centers\": [\n        \"Camel Dive Club (Naama Bay \u2014 \u20ac440-\u20ac495 OWD, \u20ac390-\u20ac440 10-dive pack; strong brand but 12-20% pricier and Naama-based away from Tiran)\",\n        \"Red Sea Diving College & Circle Divers & Emperor Divers (\u20ac380-\u20ac470 OWD; strong UK diver loyalty, fewer native languages & no interactive fee calculator)\",\n        \"Reef Oasis Dive Club (Hadaba \u2014 strong Italian/resort tie-in, standard \u20ac410+ courses)\"\n      ],\n      \"budget_excursion_brokers_and_otas\": [\n        \"Menem Tours, EgyptGate, Viator, GetYourGuide, Trip.com, Peek (advertise \u20ac35-\u20ac55 boat trips with 20-25% commission, often mixing guests onto crowded non-PADI boats and charging surprise onboard fees)\"\n      ],\n      \"sea_college_winning_wedge\": \"PADI 5\u2605 Resort safety + 3,082-review 4.9\u2605 trust + beachfront pool/House Reef right opposite Tiran + 100% transparent mid-premium EUR pricing + 8 native languages.\"\n    }\n  },\n  \"required_deliverables_from_gemini\": [\n    \"1. EXECUTIVE GROWTH STRATEGY & POSITIONING MATRIX: How Sea College wins each customer segment (Certified Divers, First-Time Beginners/DSD, PADI Course Students, VIP Private Yacht/Speedboat Groups) across our 8 language markets.\",\n    \"2. FINANCIAL & BOOKING FUNNEL MODEL (MONTHS 1, 2, 3 & 6): Starting from \u20ac1,850/month (~$2,000/mo) paid ad spend, calculate realistic clicks, CPC, CVR, leads, WhatsApp close rate, bookings, AOV, Gross Direct EUR Revenue, CAC, and ROAS \u2014 with clear scale triggers when verified CRM ROAS > 5.0x.\",\n    \"3. ZERO-DISRUPTION GOOGLE ADS ARCHITECTURE & 8-LANGUAGE CAMPAIGN BUILD: Exact campaign structure, daily budget split, bidding strategy transition (Maximize Conversions -> tCPA/tROAS via Google_Ads_OCT_Export), negative keyword lists, and native RSA headlines/descriptions for EN, RU, IT, DE, PL, FR, AR, and ZH mapped to our exact clean URLs.\",\n    \"4. PAID & ORGANIC SOCIAL + VIDEO ENGINE (IG, FB, TIKTOK, YOUTUBE, XIAOHONGSHU/VK): Weekly content pillars, exact 3-second video hooks using our underwater/boat/pool footage and named instructors, plus a high-ROAS Meta Pixel + CAPI retargeting funnel.\",\n    \"5. SEO + AEO (AI SEARCH / CHATGPT / GEMINI / PERPLEXITY) DOMINATION PLAN: How to rank our 80 tour/course pages, Dive Sites Conditions Guide, and 8-language hubs #1 for commercial Sharm diving queries and AI travel recommendations.\",\n    \"6. REVIEW VELOCITY, IN-RESORT QR & HOTEL B2B PARTNERSHIP PLAYBOOK: Turning Ivy Cyrene guests, neighbouring Al Montazah/Sharks Bay concierges, and daily boat guests into zero-CAC bookings and 100+ new Google & Tripadvisor (#26104236) reviews per quarter.\",\n    \"7. WHATSAPP + GOOGLE SHEETS CRM REVENUE OPERATIONS SOP: Exact speed-to-lead (<5 min) scripts, upsell ladders (OWD -> Stars/Mega Bundle; Dive Pack -> Nitrox/Thistlegorm/Zodiac), pre-arrival Check-In workflow, and daily 'Status=Paid' offline conversion hygiene.\",\n    \"8. WEEK-BY-WEEK 90-DAY IMPLEMENTATION ROADMAP: Concrete tasks, owners, time estimates, and weekly Friday KPI review rituals synchronized directly with the 8 tabs of our Sea College Marketing & Sales Operations Hub Google Sheet.\"\n  ]\n}")

def save_ai_output_to_sheet(task_type: str, language: str, content: str):
    """Logs any AI-generated plan, ad copy, or script into `11_Colab_AI_Outputs` in the Marketing Hub Sheet."""
    try:
        ws = mkt_wb.worksheet("11_Colab_AI_Outputs")
    except gspread.WorksheetNotFound:
        ws = mkt_wb.add_worksheet(title="11_Colab_AI_Outputs", rows=200, cols=5)
        ws.append_row(["Timestamp_UTC", "Model_Used", "Task_Type", "Language", "Generated_Output"])
    ws.append_row([
        datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
        str(SELECTED_MODEL),
        task_type,
        language,
        content[:45000]
    ])
    print("💾 Saved output to tab `11_Colab_AI_Outputs` in your Marketing & Sales Operations Hub!")

print("✅ Loaded SEA_COLLEGE_BRAIN:")
print("   • Brand:", SEA_COLLEGE_BRAIN["business_profile"]["brand_name"])
print("   • Languages:", [l["code"] for l in SEA_COLLEGE_BRAIN["final_live_website_capabilities"]["supported_languages_in_header_order"]])
print("   • Indexed Sitemap URLs:", SEA_COLLEGE_BRAIN["final_live_website_capabilities"]["seo_and_aeo_architecture"]["total_indexed_urls"])

In [ ]:
# @title 4. Live Website, 8-Language Landing URLs & Tracking Health Check { display-mode: "form" }
# @markdown Audits HTTP status, load speed, GTM (`GTM-PBVQ56XC`), GA4 (`G-99528RZN9Z`), Google Ads (`AW-16728866067`), Meta Pixel (`2980406819004765`), and `/sitemap.xml` before or after DNS cutover.
environment_to_check = "https://seacollegediveclub.com" # @param ["https://seacollegediveclub.com"]

import time
import requests

check_paths = [
    "/", "/sitemap.xml", "/prices.php", "/dive-sites-sharm-el-sheikh", "/gallery",
    "/ru/", "/it/", "/pl/", "/de/", "/fr/", "/ar/", "/zh/",
    "/tours/zodiac-speedboat-dive-sharm-el-sheikh",
    "/courses/padi-open-water-diver-sharm-el-sheikh"
]

health_rows = []
for p in check_paths:
    url = environment_to_check.rstrip("/") + p
    t0 = time.time()
    try:
        r = requests.get(url, timeout=15)
        ms = int((time.time() - t0) * 1000)
        body = r.text
        has_gtm = "GTM-PBVQ56XC" in body
        has_ads = "AW-16728866067" in body
        has_meta = "2980406819004765" in body
        health_rows.append({
            "Path": p,
            "HTTP": r.status_code,
            "Latency_ms": ms,
            "GTM": "✅" if (has_gtm or p.endswith(".xml")) else "⚠️",
            "Google_Ads": "✅" if (has_ads or p.endswith(".xml")) else "⚠️",
            "Meta_Pixel": "✅" if (has_meta or p.endswith(".xml")) else "⚠️",
        })
    except Exception as exc:
        health_rows.append({"Path": p, "HTTP": f"ERR: {exc}", "Latency_ms": -1, "GTM": "❌", "Google_Ads": "❌", "Meta_Pixel": "❌"})

display(pd.DataFrame(health_rows))

In [ ]:
# @title 5. Activate Weekly Digital Marketing & Direct-Booking Action Plan (`google.colab.ai`) { display-mode: "form" }
# @markdown Uses `google.colab.ai` + your live Google Ads & CRM numbers to generate an immediately executable **7-Day Marketing, Budget & Conversion Optimization Plan** and logs it to your Google Sheet.
monthly_paid_budget_eur = 1850 # @param {type:"number"}
seasonal_focus = "Immediate Launch — Fix Google Ads Tracking + Activate 8-Language Direct Bookings" # @param ["Immediate Launch — Fix Google Ads Tracking + Activate 8-Language Direct Bookings", "Spring/Autumn Peak — Certified Dive Packages & SS Thistlegorm Wreck", "Summer Peak — GCC/Arab Families Private Yachts & Beginner Boat DSDs", "Winter Sun — UK/DE/IT/PL/RU PADI Open Water & Mega Bundles"]
log_plan_to_sheet = True # @param {type:"boolean"}

strategy_prompt = f"""
{SEA_COLLEGE_BRAIN['prompt_metadata']['system_role_instruction']}

STRICT GROUNDING RULES:
{json.dumps(SEA_COLLEGE_BRAIN['prompt_metadata']['strict_grounding_rules'], ensure_ascii=False, indent=2)}

VERIFIED SEA COLLEGE BUSINESS, WEBSITE, PRICING & COMPETITOR DATA:
{json.dumps(SEA_COLLEGE_BRAIN, ensure_ascii=False)}

LIVE GOOGLE SHEETS & GOOGLE ADS SNAPSHOT RIGHT NOW:
{json.dumps(LIVE_SNAPSHOT, ensure_ascii=False, indent=2)}

CURRENT EXECUTION MANDATE:
- Monthly Paid Media Budget: EUR {monthly_paid_budget_eur}/month (approx {int(monthly_paid_budget_eur * 52):,} EGP/month)
- Seasonal Focus: {seasonal_focus}

Generate a crisp, immediately executable **7-Day CMO Activation & Revenue Plan** with:
1. Exact Daily Budget Split (EUR & EGP) across Google Search (8 languages) and Meta Retargeting/Video.
2. Exact Top 5 High-Impact Actions to execute in the next 7 days (referencing our exact clean URLs, Dive Calculator `/prices.php#calculator`, 30-Sec Quiz `/#quiz`, and Google Ads Zero-Reset Cutover rules).
3. Exact 3 Short-Form Video / Reels / TikTok scripts to film at Al Montazah House Reef, pool, or boat this week featuring our named instructors (Mo, Sam, Jimmy, Wesam, Saleh, Alex).
4. Exact WhatsApp Reservation Desk speed-to-lead & upsell targets to turn inquiries into `Status = Paid` bookings in our Leads CRM.
"""

weekly_plan_md = ask_gemini_colab(strategy_prompt, stream=True, render_markdown=True)
if log_plan_to_sheet:
    save_ai_output_to_sheet(f"7-Day Activation Plan ({seasonal_focus})", "multi", weekly_plan_md)

In [ ]:
# @title 6. 8-Language Google Ads RSA + Meta Ad + Reels Script Generator (Auto-Validated ≤30 / ≤90 Chars) { display-mode: "form" }
# @markdown Generates native-language Google Search RSAs (with automatic Python `<=30` / `<=90` character verification!), Meta retargeting copy, and Reels/TikTok hooks for any language & offer, and optionally stages them in your Google Sheet.
target_language = "ru - Русский (Russian)" # @param ["en - English", "ru - Русский (Russian)", "it - Italiano (Italian)", "pl - Polski (Polish)", "de - Deutsch (German)", "fr - Français (French)", "ar - العربية (Arabic)", "zh - 中文 (Chinese)"]
offer_pillar = "PADI Open Water (€400) & Mega Bundle (€600)" # @param ["PADI Open Water (€400) & Mega Bundle (€600)", "Certified 6 & 10 Dive Packages (€210 / €350) + SS Thistlegorm (€150)", "Beginner Intro Dives — House Reef (€30) & 2-Dive Boat All-In (€100)", "VIP Private Yacht (€550-€1,100) & Tiran Speedboat (€150)", "Zodiac RIB Speedboat Dive (€50) & Early Hammerheads (€70)"]
save_campaign_to_sheet = True # @param {type:"boolean"}

lang_code = target_language.split(" - ")[0].strip()
prefix = f"https://seacollegediveclub.com/{lang_code}" if lang_code != "en" else "https://seacollegediveclub.com"

ad_prompt = f"""
You are creating native, high-converting performance ads in language `{target_language}` for **Sea College Dive Club** (Official PADI 5 Star Dive Resort, Al Montazah beachfront, Sharm El Sheikh — 4.9★ across 3,082+ Google reviews & 4.9● Tripadvisor #26104236).

OFFER PILLAR: {offer_pillar}
LANGUAGE URL PREFIX: {prefix}
VERIFIED CATALOG & USPs:
{json.dumps(SEA_COLLEGE_BRAIN['official_eur_pricing_catalog'], ensure_ascii=False)}
{json.dumps(SEA_COLLEGE_BRAIN['business_profile']['reputation_and_social_proof'], ensure_ascii=False)}

Return ONLY valid JSON (no markdown fences around it) with this exact structure:
{{
  "recommended_final_url": "{prefix}/... (use exact valid slug from catalog + ?utm_source=google&utm_medium=cpc&utm_campaign={lang_code}_promo)",
  "keywords_exact_and_phrase": ["8 high-intent native keywords"],
  "negative_keywords": ["6 native negative keywords"],
  "rsa_headlines_max_30_chars": ["10 native headlines, EVERY SINGLE ONE MUST BE <= 30 characters including spaces!"],
  "rsa_descriptions_max_90_chars": ["4 native descriptions, EVERY SINGLE ONE MUST BE <= 90 characters including spaces!"],
  "meta_ig_fb_primary_text": "Native Instagram/Facebook ad copy with clear EUR price, 4.9★ social proof, €0 upfront fee & CTA",
  "reels_3sec_hook_and_shotlist": "3-second visual/audio hook + 15-second shot list at Al Montazah pool/House Reef/boat"
}}
"""

raw_ad_resp = ask_gemini_colab(ad_prompt, stream=False, render_markdown=False)
clean_json_str = raw_ad_resp.strip()
if "```" in clean_json_str:
    clean_json_str = clean_json_str.split("```json")[-1].split("```")[0].strip()

try:
    ad_pack = json.loads(clean_json_str)
    print(f"✅ Generated Native `{target_language}` Campaign Pack!\n")
    print("🔗 Final URL:", ad_pack.get("recommended_final_url"))
    print("\n🔎 Keywords:", ", ".join(ad_pack.get("keywords_exact_and_phrase", [])))
    print("🚫 Negatives:", ", ".join(ad_pack.get("negative_keywords", [])))
    print("\n📏 GOOGLE ADS HEADLINES (Strict <=30 char check):")
    for h in ad_pack.get("rsa_headlines_max_30_chars", []):
        flag = "✅" if len(h) <= 30 else "⚠️ TRIM"
        print(f"   {flag} ({len(h):2d}/30): {h}")
    print("\n📏 GOOGLE ADS DESCRIPTIONS (Strict <=90 char check):")
    for d in ad_pack.get("rsa_descriptions_max_90_chars", []):
        flag = "✅" if len(d) <= 90 else "⚠️ TRIM"
        print(f"   {flag} ({len(d):2d}/90): {d}")
    print("\n📱 META (IG/FB) AD COPY:\n", ad_pack.get("meta_ig_fb_primary_text"))
    print("\n🎬 REELS / TIKTOK 3-SEC HOOK & SHOTLIST:\n", ad_pack.get("reels_3sec_hook_and_shotlist"))
except Exception:
    display(Markdown(raw_ad_resp))

if save_campaign_to_sheet:
    save_ai_output_to_sheet(f"Ad Pack: {offer_pillar}", lang_code, raw_ad_resp)

In [ ]:
# @title 7. Automated CRM Sales Closer: Build 1-Click Localized WhatsApp Queue (`10_WhatsApp_Sales_Queue`) { display-mode: "form" }
# @markdown Reads every lead in your **Website Leads & OCT CRM**, crafts a personalized native-language WhatsApp quote + Pre-Arrival Check-In (`/prices.php#checkin`) + Google/Tripadvisor Review request, and writes 1-click links into tab `10_WhatsApp_Sales_Queue`!
use_ai_personalization_for_new_leads = True # @param {type:"boolean"}

import urllib.parse

GOOGLE_REVIEW_LINK = "https://search.google.com/local/writereview?placeid=ChIJ75cBQp5LUxQRMjrNYQx93g0"
TA_REVIEW_LINK = "https://www.tripadvisor.com/UserReview-g297555-d26104236-Sea_college_dive_club-Sharm_El_Sheikh_South_Sinai_Red_Sea_and_Sinai.html"

queue_header = [
    "Received_At", "Name", "Phone", "Lang", "Status", "Interest", "Estimated_EUR",
    "1-Click Quote WhatsApp", "1-Click Online Check-In WhatsApp", "1-Click Review Request WhatsApp"
]
queue_rows = [queue_header]

for idx, row in leads_df.iterrows():
    name = str(row.get("Name", "") or "Diver").strip()
    phone = "".join(ch for ch in str(row.get("Phone", "")) if ch.isdigit())
    if not phone:
        continue
    lang = str(row.get("Lang", "en") or "en").strip().lower()
    status = str(row.get("Status", "New") or "New").strip()
    interest = str(row.get("Interest", "Red Sea Diving") or "Red Sea Diving").strip()
    est_eur = str(row.get("Estimated_EUR", "") or "").strip()
    hotel = str(row.get("Hotel", "") or "your Sharm hotel").strip()

    if use_ai_personalization_for_new_leads and status.lower() in ("new", "") and idx < 10:
        msg_prompt = (
            f"Write a warm, concise 2-sentence WhatsApp reply in language code `{lang}` from Sea College Dive Club "
            f"(PADI 5* Resort, Sharm El Sheikh) to `{name}` who inquired about `{interest}` "
            f"(Estimated EUR: `{est_eur or 'confirming quote'}`, Hotel: `{hotel}`). "
            f"Mention €0 upfront fee and free hotel pickup, and ask to confirm their preferred date. Plain text only."
        )
        quote_text = ask_gemini_colab(msg_prompt, stream=False, render_markdown=False).strip()
    else:
        price_str = f" (€{est_eur} total, €0 upfront booking fee)" if est_eur else " (€0 upfront booking fee)"
        quote_text = f"Hi {name}! 🤿 Sea College Dive Club (PADI 5★ Resort, Sharm El Sheikh) here — we received your {interest} request{price_str}. Shall we lock your spot & free pickup from {hotel}?"

    checkin_text = (
        f"Hi {name}! 🎉 Your Sea College Dive Club booking is confirmed! Complete your 2-min Online Diver Check-In "
        f"before arrival so your gear is ready: https://seacollegediveclub.com/prices.php#checkin"
    )
    review_text = (
        f"Thank you for diving with Sea College Dive Club today, {name}! 🐠 A quick 30-sec review helps our team hugely:\n"
        f"⭐ Google: {GOOGLE_REVIEW_LINK}\n🦉 Tripadvisor: {TA_REVIEW_LINK}"
    )

    q_url = f"https://wa.me/{phone}?text={urllib.parse.quote(quote_text)}"
    c_url = f"https://wa.me/{phone}?text={urllib.parse.quote(checkin_text)}"
    r_url = f"https://wa.me/{phone}?text={urllib.parse.quote(review_text)}"

    queue_rows.append([
        str(row.get("Received_At", "")), name, phone, lang, status, interest, est_eur,
        f'=HYPERLINK("{q_url}","📲 Send Quote")',
        f'=HYPERLINK("{c_url}","📋 Send Check-In")',
        f'=HYPERLINK("{r_url}","⭐ Request Review")',
    ])

try:
    q_ws = mkt_wb.worksheet("10_WhatsApp_Sales_Queue")
    q_ws.clear()
except gspread.WorksheetNotFound:
    q_ws = mkt_wb.add_worksheet(title="10_WhatsApp_Sales_Queue", rows=max(50, len(queue_rows) + 10), cols=10)

q_ws.update(range_name=f"A1:J{len(queue_rows)}", values=queue_rows, value_input_option="USER_ENTERED")
print(f"✅ Built {len(queue_rows) - 1} personalized 1-click WhatsApp links in tab `10_WhatsApp_Sales_Queue`!")

In [ ]:
# @title 8. Interactive Chat with Your Sea College AI CMO (`google.colab.ai`) { display-mode: "form" }
# @markdown Ask any custom marketing, pricing, B2B hotel partnership, SEO/AEO, or Google Ads question — grounded automatically in your live spreadsheets and website catalog!
your_question = "Create a high-converting weekend WhatsApp & Instagram flash promo for guests currently staying in Sharm El Sheikh hotels to book our €100 All-Inclusive 2-Dive Boat Intro or €150 Private Tiran Speedboat." # @param {type:"string"}
save_answer_to_sheet = True # @param {type:"boolean"}

if "CHAT_HISTORY" not in globals():
    CHAT_HISTORY = []

history_block = "\n".join([f"User: {u}\nCMO: {a}" for u, a in CHAT_HISTORY[-4:]])
custom_prompt = f"""
{SEA_COLLEGE_BRAIN['prompt_metadata']['system_role_instruction']}

SEA COLLEGE VERIFIED DATA:
{json.dumps(SEA_COLLEGE_BRAIN, ensure_ascii=False)}

LIVE SHEETS & ADS METRICS:
{json.dumps(LIVE_SNAPSHOT, ensure_ascii=False)}

RECENT CONVERSATION CONTEXT:
{history_block}

USER REQUEST:
{your_question}
"""

cmo_reply = ask_gemini_colab(custom_prompt, stream=True, render_markdown=True)
CHAT_HISTORY.append((your_question, cmo_reply))
if save_answer_to_sheet:
    save_ai_output_to_sheet("Custom CMO Chat", "multi", f"Q: {your_question}\n\n{cmo_reply}")